# Host-telemetry pipeline (Security-Datasets) — self-contained

Runs standalone. Order: **port → SETUP → A → B → C → D → E → F → G**.

- SETUP defines the baseline vocabulary/ATT&CK/LLM that the CIC-IDS notebook supplied via earlier cells. It uses your local files (`../data/attck/...`, `../models/...`) if present, and falls back so data + KG + eval still run.
- CELL B is idempotent (safe to re-run).
- CELL F: on this atomic dataset report the **malicious-process** result, not blanket accuracy.

In [1]:
# ============================================================================
#  PORT CELL — Security-Datasets (OTRF) Sysmon JSON  ->  semantic alerts
#  Replaces the CIC-IDS netflow loader + build_semantic_alert.
#  Dataset: credential_access / cmd_lsass_memory_dumpert_syscalls  (T1003.001)
#
#  Key design change vs CIC-IDS:
#    CIC-IDS  : 1 alert per flow row  (statistical, no relations -> RAG failed)
#    Here     : 1 alert per PROCESS (grouped by ProcessGuid), verbalizing the
#               behaviors that process performed across event types.
#               This gives the LLM real subject-relation-object content.
# ============================================================================

import json, os, urllib.request, zipfile, io
from collections import defaultdict
import pandas as pd

# ---------------------------------------------------------------------------
# 1. DOWNLOAD + LOAD  (JSON-lines: one event per line)
# ---------------------------------------------------------------------------
URL = ("https://raw.githubusercontent.com/OTRF/Security-Datasets/master/"
       "datasets/atomic/windows/credential_access/host/"
       "cmd_lsass_memory_dumpert_syscalls.zip")
GROUND_TRUTH_TECHNIQUE = "T1003.001"   # OS Credential Dumping: LSASS Memory

def load_secdataset(url=URL, local_dir="data/secdatasets"):
    os.makedirs(local_dir, exist_ok=True)
    raw = urllib.request.urlopen(url).read()
    zf = zipfile.ZipFile(io.BytesIO(raw))
    name = [n for n in zf.namelist() if n.endswith(".json")][0]
    zf.extractall(local_dir)
    path = os.path.join(local_dir, name)
    # JSON-lines -> DataFrame (NOT plain read_json; must pass lines=True)
    df = pd.read_json(path, lines=True)
    print(f"Loaded {len(df)} events from {name}")
    print("EventID distribution:\n", df["EventID"].value_counts().to_dict())
    return df

# ---------------------------------------------------------------------------
# 2. GROUP EVENTS BY PROCESS  (the unit of analysis)
#    A process is identified by ProcessGuid (Sysmon) or SourceProcessGUID (EID10).
# ---------------------------------------------------------------------------
def _clean(v):
    # pandas turns missing JSON fields into float('nan'); normalize to None
    if v is None: return None
    if isinstance(v, float): return None
    s = str(v).strip()
    return s if s and s.lower() != "nan" else None

def _guid(ev):
    return _clean(ev.get("ProcessGuid")) or _clean(ev.get("SourceProcessGUID"))

def _image(ev):
    return _clean(ev.get("Image")) or _clean(ev.get("SourceImage"))

def group_by_process(df):
    procs = defaultdict(lambda: {
        "image": None, "command_line": None,
        "parent_image": None, "parent_command_line": None,
        "user": None, "integrity": None, "hostname": None,
        "loaded_dlls": [], "accessed": [], "registry": [],
        "files_created": [], "network": [],
    })
    rows = df.to_dict("records")
    # first pass: identity from ProcessCreate (EID 1)
    for ev in rows:
        if ev.get("EventID") == 1:
            g = _clean(ev.get("ProcessGuid"))
            if not g: continue
            p = procs[g]
            p["image"]               = _image(ev)
            p["command_line"]        = _clean(ev.get("CommandLine"))
            p["parent_image"]        = _clean(ev.get("ParentImage"))
            p["parent_command_line"] = _clean(ev.get("ParentCommandLine"))
            p["user"]                = _clean(ev.get("User"))
            p["integrity"]           = _clean(ev.get("IntegrityLevel"))
            p["hostname"]            = _clean(ev.get("Hostname"))
    # second pass: behaviors
    for ev in rows:
        eid = ev.get("EventID")
        g = _guid(ev)
        if not g: continue
        p = procs[g]
        if p["image"] is None:
            p["image"] = _image(ev)          # backfill identity if no EID1
        if p["hostname"] is None:
            p["hostname"] = ev.get("Hostname")
        if eid == 7:                          # ImageLoad
            dll = ev.get("ImageLoaded")
            if dll: p["loaded_dlls"].append(dll)
        elif eid == 10:                       # ProcessAccess  <-- the LSASS signal
            if p["image"] is None:            # name the accessor from SourceImage
                p["image"] = _clean(ev.get("SourceImage"))
            p["accessed"].append({
                "target": ev.get("TargetImage"),
                "access": ev.get("GrantedAccess"),
            })
        elif eid in (12, 13, 14):             # Registry
            p["registry"].append({
                "key": ev.get("TargetObject"),
                "type": ev.get("EventType"),
            })
        elif eid == 11:                       # FileCreate
            f = ev.get("TargetFilename")
            if f: p["files_created"].append(f)
        elif eid in (3, 5156):                # Network
            p["network"].append({
                "dst": ev.get("DestinationIp") or ev.get("DestAddress"),
                "port": ev.get("DestinationPort") or ev.get("DestPort"),
            })
    return procs

# ---------------------------------------------------------------------------
# 3. SEMANTIC ALERT BUILDER  (drop-in replacement for the netflow version)
#    Produces behavioral, ATT&CK-flavored prose -> closes the embedding gap.
# ---------------------------------------------------------------------------
SUSPICIOUS_ACCESS = {"0x1fffff", "0x1010", "0x1410", "0x143a", "0x1438"}

def _basename(path):
    return str(path).replace("\\", "/").split("/")[-1] if path else "unknown"

def build_semantic_alert(proc):
    """proc = one value dict from group_by_process()."""
    img  = _basename(proc["image"])
    sent = []

    # identity / lineage
    if proc["parent_image"]:
        sent.append(f"The process {img} was spawned by "
                    f"{_basename(proc['parent_image'])}.")
    else:
        sent.append(f"Observed activity from the process {img}.")
    if proc["command_line"]:
        sent.append(f"It executed with command line: {proc['command_line']}.")
    if proc["user"]:
        sent.append(f"Running as {proc['user']} "
                    f"at {proc['integrity'] or 'unknown'} integrity.")

    # credential-access behavior (the technique signal)
    lsass_hits = [a for a in proc["accessed"]
                  if "lsass" in str(a["target"]).lower()]
    if lsass_hits:
        acc = lsass_hits[0]["access"]
        flag = " with full access rights" if acc in SUSPICIOUS_ACCESS else ""
        sent.append(f"It opened a handle to the LSASS process memory "
                    f"(granted access {acc}){flag}, "
                    f"a behavior associated with credential extraction.")
    elif proc["accessed"]:
        tgts = {_basename(a["target"]) for a in proc["accessed"]}
        sent.append("It accessed the memory of other processes: "
                    f"{', '.join(sorted(tgts))}.")

    # supporting behaviors
    if proc["loaded_dlls"]:
        dlls = {_basename(d) for d in proc["loaded_dlls"]}
        notable = [d for d in dlls if d.lower() in
                   {"ntdll.dll","dbghelp.dll","dbgcore.dll","samlib.dll"}]
        if notable:
            sent.append("It loaded libraries commonly used for memory "
                        f"manipulation: {', '.join(sorted(notable))}.")
    if proc["registry"]:
        keys = {_basename(r["key"]) for r in proc["registry"] if r["key"]}
        if keys:
            sent.append("It modified registry values including "
                        f"{', '.join(list(keys)[:3])}.")
    if proc["files_created"]:
        sent.append(f"It created {len(proc['files_created'])} file(s) on disk.")
    if proc["network"]:
        sent.append(f"It made {len(proc['network'])} outbound network "
                    "connection(s).")

    return " ".join(sent)

# ---------------------------------------------------------------------------
# 4. BUILD THE ALERT TABLE  (this is what feeds embedding/community/RAG)
# ---------------------------------------------------------------------------
def build_alert_dataframe(df):
    procs = group_by_process(df)
    records = []
    for guid, p in procs.items():
        # skip pure-noise system processes with no behavior captured
        if not (p["accessed"] or p["loaded_dlls"] or p["registry"]
                or p["command_line"] or p["files_created"]):
            continue
        records.append({
            "process_guid": guid,
            "image": _basename(p["image"]),
            "semantic_alert": build_semantic_alert(p),
            "ground_truth_technique": GROUND_TRUTH_TECHNIQUE,
        })
    return pd.DataFrame(records)

# ---------------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    df = load_secdataset()
    alerts = build_alert_dataframe(df)
    print(f"\nBuilt {len(alerts)} process-level semantic alerts.\n")
    for _, r in alerts.iterrows():
        print(f"[{r['image']}]  ({str(r['process_guid'])[:18]}...)")
        print("  " + r["semantic_alert"])
        print()

Loaded 118 events from cmd_lsass_memory_dumpert_syscalls_2020-10-1822561997.json
EventID distribution:
 {10: 44, 13: 23, 7: 21, 4658: 6, 4663: 3, 4656: 3, 4690: 3, 12: 3, 5156: 2, 5158: 2, 11: 2, 4689: 1, 4703: 1, 4688: 1, 1102: 1, 5: 1, 1: 1}

Built 8 process-level semantic alerts.

[Outflank-Dumpert.exe]  ({39e4a257-004e-5f8...)
  The process Outflank-Dumpert.exe was spawned by cmd.exe. It executed with command line: Outflank-Dumpert.exe. Running as WORKSTATION5\wardog at High integrity. It opened a handle to the LSASS process memory (granted access 0x1fffff) with full access rights, a behavior associated with credential extraction. It loaded libraries commonly used for memory manipulation: dbgcore.dll, dbghelp.dll, ntdll.dll. It created 1 file(s) on disk.

[svchost.exe]  ({39e4a257-f132-5f8...)
  Observed activity from the process svchost.exe. It accessed the memory of other processes: conhost.exe, svchost.exe.

[Explorer.EXE]  ({39e4a257-f1b3-5f8...)
  Observed activity from the pr

In [2]:
# =====================================================================
#  SETUP — base definitions (makes this notebook self-contained)
#  ---------------------------------------------------------------------
#  In the CIC-IDS notebook these came from earlier cells. Here we define
#  the baseline inline so A-G run standalone. Paths default to the CIC-IDS
#  layout but FALL BACK gracefully if the file/model isn't present, so the
#  data + KG + eval stages still run without the local LLM/ATT&CK bundle.
# =====================================================================
import os, json, re
from pathlib import Path
from collections import Counter
import pandas as pd, numpy as np, networkx as nx

RANDOM_SEED = 42
RESULTS_DIR = Path('../results'); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ATTCK_PATH  = Path('../data/attck/enterprise-attack.json')
MODEL_PATH  = '../models/qwen2.5-3b-instruct-q4_k_m.gguf'

# ---- baseline relation vocabulary (the CIC-IDS netflow set) ----
SECURITY_RELATIONS = [
    'PERFORMS_RECONNAISSANCE', 'PERFORMS_PORT_SCAN', 'BRUTE_FORCES_CREDENTIAL',
    'ACCESS_CREDENTIALS', 'EXPLOITS_VULNERABILITY', 'ESTABLISHES_C2',
    'PERFORMS_BEACONING', 'CAUSES_DENIAL_OF_SERVICE', 'MOVES_LATERALLY',
    'EXFILTRATES_DATA', 'EXECUTES_PAYLOAD',
]

RELATION_GUIDE = """
Relation definitions (use exactly as written):
  PERFORMS_RECONNAISSANCE  : active discovery, general service enumeration and scanning
  PERFORMS_PORT_SCAN       : focused port/protocol probing across multiple ports or hosts
  BRUTE_FORCES_CREDENTIAL  : repeated authentication attempts against a service
  ACCESS_CREDENTIALS       : theft or collection of credentials
  EXPLOITS_VULNERABILITY   : exploitation of a software or configuration flaw
  ESTABLISHES_C2           : outbound communication to a command-and-control channel
  PERFORMS_BEACONING       : periodic callback/beacon patterns indicative of persistent C2
  CAUSES_DENIAL_OF_SERVICE : flooding or resource exhaustion of a target service
  MOVES_LATERALLY          : accessing internal hosts after initial compromise
  EXFILTRATES_DATA         : transferring data out of the environment
  EXECUTES_PAYLOAD         : running code or commands on a target
""".strip()

RELATION_TO_TECHNIQUES = {
    'PERFORMS_RECONNAISSANCE':  ['T1046', 'T1595', 'T1590'],
    'PERFORMS_PORT_SCAN':       ['T1046', 'T1595'],
    'BRUTE_FORCES_CREDENTIAL':  ['T1110', 'T1110.001', 'T1110.003'],
    'ACCESS_CREDENTIALS':       ['T1555', 'T1078', 'T1110'],
    'EXPLOITS_VULNERABILITY':   ['T1190', 'T1203'],
    'ESTABLISHES_C2':           ['T1071', 'T1071.001', 'T1071.004'],
    'PERFORMS_BEACONING':       ['T1071', 'T1071.004'],
    'CAUSES_DENIAL_OF_SERVICE': ['T1498', 'T1499'],
    'MOVES_LATERALLY':          ['T1021'],
    'EXFILTRATES_DATA':         ['T1041', 'T1048'],
    'EXECUTES_PAYLOAD':         ['T1059', 'T1203'],
}

RELATION_TO_TACTIC_SCOPE = {
    'PERFORMS_RECONNAISSANCE':  'discovery',
    'PERFORMS_PORT_SCAN':       'discovery',
    'BRUTE_FORCES_CREDENTIAL':  'credential-access',
    'ACCESS_CREDENTIALS':       'credential-access',
    'EXPLOITS_VULNERABILITY':   'initial-access',
    'ESTABLISHES_C2':           'command-and-control',
    'PERFORMS_BEACONING':       'command-and-control',
    'CAUSES_DENIAL_OF_SERVICE': 'impact',
    'MOVES_LATERALLY':          'lateral-movement',
    'EXFILTRATES_DATA':         'exfiltration',
    'EXECUTES_PAYLOAD':         'execution',
}

TRIPLE_SCHEMA = {
    'type': 'object',
    'properties': {
        'triples': {
            'type': 'array', 'minItems': 1, 'maxItems': 4,
            'items': {
                'type': 'object',
                'properties': {
                    'subject':  {'type': 'string'},
                    'relation': {'type': 'string', 'enum': SECURITY_RELATIONS},
                    'target':   {'type': 'string'},
                },
                'required': ['subject', 'relation', 'target'],
            },
        },
    },
    'required': ['triples'],
}

def normalise_entity(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9_]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text[:80]

# ---- ATT&CK techniques: load local STIX bundle if available ----
attck_techniques = {}
if ATTCK_PATH.exists():
    with open(ATTCK_PATH, 'r', encoding='utf-8') as f:
        bundle = json.load(f)
    for obj in bundle.get('objects', []):
        if obj.get('type') != 'attack-pattern': continue
        if obj.get('revoked') or obj.get('x_mitre_deprecated'): continue
        tid = next((r.get('external_id') for r in obj.get('external_references', [])
                    if r.get('source_name') == 'mitre-attack'), None)
        if not tid: continue
        tactics = [p.get('phase_name','').capitalize()
                   for p in obj.get('kill_chain_phases', []) if p.get('phase_name')]
        attck_techniques[tid] = {'name': obj.get('name','Unknown'),
                                 'tactics': tactics,
                                 'description': obj.get('description','')}
    print(f"Loaded {len(attck_techniques)} ATT&CK techniques from {ATTCK_PATH}")
else:
    # minimal fallback so KG + eval still run without the bundle
    attck_techniques = {
        'T1003.001': {'name': 'LSASS Memory', 'tactics': ['Credential-access'], 'description': ''},
        'T1003':     {'name': 'OS Credential Dumping', 'tactics': ['Credential-access'], 'description': ''},
        'T1055':     {'name': 'Process Injection', 'tactics': ['Defense-evasion'], 'description': ''},
        'T1059':     {'name': 'Command and Scripting Interpreter', 'tactics': ['Execution'], 'description': ''},
        'T1112':     {'name': 'Modify Registry', 'tactics': ['Defense-evasion'], 'description': ''},
        'T1574':     {'name': 'Hijack Execution Flow', 'tactics': ['Defense-evasion'], 'description': ''},
    }
    print(f"ATT&CK bundle not found at {ATTCK_PATH} — using {len(attck_techniques)}-technique fallback.")

# ---- LLM: load local gguf if present; else leave llm undefined ----
try:
    from llama_cpp import Llama, LlamaGrammar
    if os.path.exists(MODEL_PATH):
        llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=-1,
                    n_threads=8, n_batch=256, verbose=False, seed=RANDOM_SEED)
        print(f"LLM loaded: {MODEL_PATH}")
    else:
        print(f"Model not found at {MODEL_PATH} — CELL C/D need it; "
              f"data/KG/eval stages run without it.")
except Exception as e:
    print(f"llama_cpp unavailable ({e}); CELL C/D need the LLM.")

print("Setup complete.")


Loaded 691 ATT&CK techniques from ../data/attck/enterprise-attack.json


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM loaded: ../models/qwen2.5-3b-instruct-q4_k_m.gguf
Setup complete.


In [3]:
# =====================================================================
#  CELL A — EMBED + COMMUNITY DETECTION  (host alerts)
#  ---------------------------------------------------------------------
#  CHANGE vs CIC-IDS: the atomic dataset has only ~8 alerts, far too few
#  for util.community_detection to form real clusters. We therefore lower
#  min_community_size and FALL BACK to "one community = one process" when
#  clustering is degenerate. On APT29 (hundreds of alerts) remove the
#  fallback and use clustering as taught. Treat community detection here
#  as "verified runs", not as a tuned result.
# =====================================================================
from sentence_transformers import SentenceTransformer, util
import numpy as np, torch

embedder = SentenceTransformer('all-MiniLM-L6-v2')

alert_texts_all = alerts['semantic_alert'].tolist()
embeddings = embedder.encode(alert_texts_all, convert_to_tensor=True,
                             show_progress_bar=False)

N = len(alerts)
MIN_COMMUNITY_SIZE = 2 if N < 30 else 3      # relax for tiny atomic sets
communities = util.community_detection(
    embeddings, min_community_size=MIN_COMMUNITY_SIZE, threshold=0.55)

comm_ids = [-1] * N
for cid, members in enumerate(communities):
    for idx in members:
        comm_ids[idx] = cid

# Fallback: if clustering collapsed everything to noise, treat each alert
# as its own singleton community so downstream stages still execute.
if all(c == -1 for c in comm_ids):
    print("Clustering degenerate at this scale — using per-process singletons.")
    comm_ids = list(range(N))

alerts['community_id'] = comm_ids
assigned = (alerts['community_id'] != -1).sum()
print(f"{assigned}/{N} alerts assigned to {len(set(c for c in comm_ids if c!=-1))} communities")
print(alerts['community_id'].value_counts(dropna=False).head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


7/8 alerts assigned to 1 communities
community_id
 0    7
-1    1
Name: count, dtype: int64


In [4]:
# =====================================================================
#  CELL B — EXTEND RELATION VOCAB so this technique is REACHABLE
#  ---------------------------------------------------------------------
#  Adds host relations and points credential-dumping at T1003 / T1003.001.
#  IDEMPOTENT: safe to re-run (uses a sentinel + rebuilds GUIDE from base).
#  Grammar rebuild is skipped gracefully if llama_cpp / LlamaGrammar
#  is unavailable (data/KG/eval stages don't need it).
# =====================================================================
HOST_RELATIONS = [
    'DUMPS_CREDENTIALS',     # LSASS / SAM memory access
    'INJECTS_INTO_PROCESS',  # remote thread / handle abuse
    'LOADS_SUSPICIOUS_DLL',  # dbghelp/dbgcore etc.
    'MODIFIES_REGISTRY',     # persistence / config
    'SPAWNS_PROCESS',        # process lineage
]
HOST_GUIDE = """
  DUMPS_CREDENTIALS        : accessing LSASS/SAM process memory to extract credentials
  INJECTS_INTO_PROCESS     : opening a handle to another process to inject or read memory
  LOADS_SUSPICIOUS_DLL     : loading libraries commonly abused for memory manipulation
  MODIFIES_REGISTRY        : creating or setting registry values
  SPAWNS_PROCESS           : a parent process creating a child process""".rstrip()

# extend the enum, de-duplicated (safe to re-run)
for r in HOST_RELATIONS:
    if r not in SECURITY_RELATIONS:
        SECURITY_RELATIONS.append(r)

# rebuild the guide from a stored base, so re-running never stacks duplicates
if '_RELATION_GUIDE_BASE' not in globals():
    _RELATION_GUIDE_BASE = RELATION_GUIDE
RELATION_GUIDE = _RELATION_GUIDE_BASE + "\n" + HOST_GUIDE

RELATION_TO_TECHNIQUES.update({
    'DUMPS_CREDENTIALS':    ['T1003.001', 'T1003', 'T1555'],
    'INJECTS_INTO_PROCESS': ['T1055', 'T1055.001'],
    'LOADS_SUSPICIOUS_DLL': ['T1574', 'T1055'],
    'MODIFIES_REGISTRY':    ['T1112', 'T1547.001'],
    'SPAWNS_PROCESS':       ['T1059', 'T1106'],
})
RELATION_TO_TACTIC_SCOPE.update({
    'DUMPS_CREDENTIALS':    'credential-access',
    'INJECTS_INTO_PROCESS': 'defense-evasion',
    'LOADS_SUSPICIOUS_DLL': 'defense-evasion',
    'MODIFIES_REGISTRY':    'defense-evasion',
    'SPAWNS_PROCESS':       'execution',
})

# keep schema enum in sync; rebuild grammar only if the LLM stack is present
TRIPLE_SCHEMA['properties']['triples']['items']['properties']['relation']['enum'] = SECURITY_RELATIONS
grammar = None
try:
    from llama_cpp import LlamaGrammar
    grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
    print(f"Grammar rebuilt — {len(SECURITY_RELATIONS)} relations "
          f"({len(HOST_RELATIONS)} host-specific added)")
except Exception as e:
    print(f"Grammar not built (llama_cpp unavailable: {e}). "
          f"Enum updated to {len(SECURITY_RELATIONS)} relations; "
          f"CELL C/D will need the LLM.")


Grammar rebuilt — 16 relations (5 host-specific added)


In [5]:
# =====================================================================
#  CELL C — HOST-ADAPTED TRIPLE EXTRACTION  (priority-aware prompt)
#  ---------------------------------------------------------------------
#  FIX vs prior run: the 3B model extracted every supporting detail and
#  let LOADS_SUSPICIOUS_DLL outnumber the real signal, and it hallucinated
#  DUMPS_CREDENTIALS for processes that merely accessed another proc's
#  memory. The prompt now (1) tells the model to put the SINGLE most
#  security-significant behavior FIRST, and (2) restricts DUMPS_CREDENTIALS
#  to cases where LSASS/SAM is explicitly the target.
# =====================================================================
def extract_triples(alert_texts: list, community_id: int) -> list:
    if 'llm' not in globals():
        raise RuntimeError('LLM not initialized. Run the SETUP cell first.')
    block = '\n'.join(f'- {t}' for t in alert_texts[:6])
    assert ' Label' not in block and 'Technique' not in block, \
        f"Potential label leakage in community {community_id}"

    prompt = f"""[INST] You are a cybersecurity analyst triaging host telemetry.
Read the alert(s) below and extract 1 to 4 semantic triples.

{RELATION_GUIDE}

CRITICAL RULES:
1. The FIRST triple must describe the SINGLE most security-significant
   behavior in the alert. Order the rest by decreasing severity.
2. Use DUMPS_CREDENTIALS ONLY when the alert explicitly says a process
   opened/accessed LSASS or SAM memory. Accessing some OTHER process's
   memory is INJECTS_INTO_PROCESS, never DUMPS_CREDENTIALS.
3. Loading a DLL is a SUPPORTING detail (LOADS_SUSPICIOUS_DLL) — never the
   primary behavior if a credential, injection, or execution action exists.
4. subject/target must name what the alert names (e.g. 'dumpert_exe',
   'lsass_memory'). No generic placeholders.

Example A — alert mentions opening a handle to LSASS memory for credential extraction:
{{"triples": [
  {{"subject": "dumpert_exe", "relation": "DUMPS_CREDENTIALS",    "target": "lsass_memory"}},
  {{"subject": "dumpert_exe", "relation": "LOADS_SUSPICIOUS_DLL", "target": "dbghelp_dll"}},
  {{"subject": "cmd_exe",     "relation": "SPAWNS_PROCESS",       "target": "dumpert_exe"}}
]}}

Example B — alert says a process accessed the memory of OTHER processes (not LSASS):
{{"triples": [
  {{"subject": "explorer_exe", "relation": "INJECTS_INTO_PROCESS", "target": "cmd_exe"}}
]}}

ALERT(S):
{block}

Return ONLY the JSON object. [/INST]"""

    try:
        out = llm(prompt, grammar=grammar, max_tokens=512, temperature=0.0)
        raw = out['choices'][0]['text'].strip()
        parsed = json.loads(raw)
        triples = parsed.get('triples', [])
        for t in triples:
            t['subject'] = normalise_entity(t.get('subject', ''))
            t['target']  = normalise_entity(t.get('target', ''))
        return triples
    except Exception as e:
        print(f'  [Community {community_id}] parse failed: {e}')
        return []


In [6]:
# =====================================================================
#  CELL D — RUN EXTRACTION  (unit = process at atomic scale)
#  ---------------------------------------------------------------------
#  FINDING (atomic scale): semantic community detection merged behaviorally
#  distinct processes (Dumpert + benign Explorer/csrss/cmd) into ONE
#  community because their alert texts share the phrase "accessed the memory
#  of other processes". Reading that merged community as a block washed out
#  the LSASS credential-dumping signal -> the LLM saw generic process access
#  and returned INJECTS_INTO_PROCESS. This is the SAME signal-averaging
#  failure observed on CIC-IDS, reproduced on host data.
#
#  FIX: at small N, extract per PROCESS (one alert -> one triple set) so the
#  malicious process is read in isolation. At larger scale (APT29) set
#  EXTRACT_UNIT='community' to use clustering as taught. The unit is an
#  explicit, reportable choice — not silent tuning.
# =====================================================================
from pathlib import Path

# ---------------------------------------------------------------------
#  DETERMINISTIC VALIDATOR (LLM proposes, code verifies against the text)
#  The 3B model hallucinates DUMPS_CREDENTIALS for processes that merely
#  accessed another process's memory (no LSASS). We enforce the grounding
#  rule in code: a credential-dumping triple is kept only if the source
#  alert text actually names LSASS or SAM; otherwise it is downgraded to
#  INJECTS_INTO_PROCESS (what those processes truly did). Standard LLM+rules
#  practice; makes extraction auditable and is itself a thesis finding.
# ---------------------------------------------------------------------
def validate_triples(triples, alert_text):
    text = alert_text.lower()
    has_lsass = ('lsass' in text or ' sam ' in text or 'sam database' in text)
    accessed_mem = ('accessed the memory' in text) or ('handle to' in text)
    fixed = []
    for t in triples:
        rel = t.get('relation', '')
        if rel in ('DUMPS_CREDENTIALS', 'ACCESS_CREDENTIALS') and not has_lsass:
            t = {**t,
                 'relation': 'INJECTS_INTO_PROCESS' if accessed_mem else 'EXECUTES_PAYLOAD',
                 'downgraded_from': rel}
        fixed.append(t)
    return fixed

EXTRACT_UNIT = 'process' if len(alerts) < 30 else 'community'
print(f"Extraction unit: {EXTRACT_UNIT}  (N={len(alerts)} alerts)\n")

community_df = alerts[alerts['community_id'] != -1].copy()
community_triples = {}

if EXTRACT_UNIT == 'process':
    # one extraction per process; key by a stable per-process id so the KG /
    # eval cells (which iterate community_triples + community_df['community_id'])
    # keep working unchanged. We overwrite community_id to be per-process.
    alerts['community_id'] = range(len(alerts))
    community_df = alerts.copy()
    for _, row in community_df.iterrows():
        cid = int(row['community_id'])
        triples = extract_triples([row['semantic_alert']], cid)
        triples = validate_triples(triples, row['semantic_alert'])
        community_triples[str(cid)] = triples
        rels = [t['relation'] for t in triples]
        n_fixed = sum('downgraded_from' in t for t in triples)
        flag = f"  (downgraded {n_fixed})" if n_fixed else ""
        print(f"  [{row['image']:<22}] cid={cid}: {len(triples)} triples  {rels}{flag}")
else:
    for cid, group in community_df.groupby('community_id'):
        texts = group['semantic_alert'].tolist()
        triples = extract_triples(texts, int(cid))
        triples = validate_triples(triples, ' '.join(texts))
        community_triples[str(cid)] = triples
        rels = [t['relation'] for t in triples]
        print(f"  Community {cid}: {len(triples)} triples  {rels}")

# persist
try:
    with open(Path(RESULTS_DIR) / 'host_community_triples.json', 'w', encoding='utf-8') as f:
        json.dump(community_triples, f, indent=2)
except Exception:
    pass


Extraction unit: process  (N=8 alerts)

  [Outflank-Dumpert.exe  ] cid=0: 4 triples  ['LOADS_SUSPICIOUS_DLL', 'LOADS_SUSPICIOUS_DLL', 'LOADS_SUSPICIOUS_DLL', 'DUMPS_CREDENTIALS']
  [svchost.exe           ] cid=1: 4 triples  ['INJECTS_INTO_PROCESS', 'INJECTS_INTO_PROCESS', 'LOADS_SUSPICIOUS_DLL', 'LOADS_SUSPICIOUS_DLL']
  [Explorer.EXE          ] cid=2: 4 triples  ['MODIFIES_REGISTRY', 'MODIFIES_REGISTRY', 'MODIFIES_REGISTRY', 'LOADS_SUSPICIOUS_DLL']
  [svchost.exe           ] cid=3: 4 triples  ['LOADS_SUSPICIOUS_DLL', 'SPAWNS_PROCESS', 'EXFILTRATES_DATA', 'EXECUTES_PAYLOAD']
  [cmd.exe               ] cid=4: 3 triples  ['INJECTS_INTO_PROCESS', 'LOADS_SUSPICIOUS_DLL', 'INJECTS_INTO_PROCESS']  (downgraded 1)
  [conhost.exe           ] cid=5: 4 triples  ['INJECTS_INTO_PROCESS', 'LOADS_SUSPICIOUS_DLL', 'INJECTS_INTO_PROCESS', 'LOADS_SUSPICIOUS_DLL']  (downgraded 1)
  [csrss.exe             ] cid=6: 4 triples  ['INJECTS_INTO_PROCESS', 'INJECTS_INTO_PROCESS', 'LOADS_SUSPICIOUS_DLL', 'MODIFIE

In [7]:
# =====================================================================
#  CELL E — KNOWLEDGE GRAPH  (reuses your two-layer design)
#  ---------------------------------------------------------------------
#  Identical construction to CIC-IDS cell 32: behavioral layer from triples,
#  ATT&CK layer from attck_techniques, bridge edges via RELATION_TO_TECHNIQUES.
# =====================================================================
import networkx as nx
G = nx.DiGraph()
edge_weights, edge_communities = {}, {}
total_triples = invalid_triples = 0

for cid, triples in community_triples.items():
    for t in triples:
        s, r, o = (str(t.get(k, '')).strip() for k in ('subject', 'relation', 'target'))
        if not (s and r and o):
            invalid_triples += 1
            continue
        total_triples += 1
        key = (s, r, o)
        edge_weights[key] = edge_weights.get(key, 0) + 1
        edge_communities.setdefault(key, []).append(cid)

for (src, rel, tgt), w in edge_weights.items():
    G.add_node(src, layer='behavioral')
    G.add_node(tgt, layer='behavioral')
    G.add_edge(src, tgt, relation=rel, weight=w, layer='behavioral',
               communities=','.join(edge_communities[(src, rel, tgt)]))

if 'attck_techniques' not in globals():
    raise RuntimeError('Run the ATT&CK load cell first.')
for tid, info in attck_techniques.items():
    if tid not in G:
        G.add_node(tid, layer='attck', name=info.get('name', ''),
                   tactic=(info.get('tactics') or [''])[0],
                   description=info.get('description', ''))

# bridge behavioral relations -> candidate techniques
bridges = 0
for (src, rel, tgt) in edge_weights:
    for tid in RELATION_TO_TECHNIQUES.get(rel, []):
        if tid in G:
            G.add_edge(tgt, tid, relation='MAPS_TO', layer='bridge')
            bridges += 1
print(f"KG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges | "
      f"{total_triples} triples ({invalid_triples} invalid), {bridges} bridges")

KG: 710 nodes, 68 edges | 30 triples (0 invalid), 61 bridges


In [8]:
# =====================================================================
#  CELL F — EVALUATE  (severity-priority instead of majority vote)
#  ---------------------------------------------------------------------
#  FIX vs prior run: majority-vote let frequent low-signal relations
#  (LOADS_SUSPICIOUS_DLL x3) bury the real one. Real triage prioritizes by
#  severity, not frequency. We rank relations and the predicted technique
#  set comes from the HIGHEST-PRIORITY relation the model emitted for that
#  process. CELL C is now prompted to emit that relation FIRST, so this also
#  rewards correct ordering.
# =====================================================================
from collections import Counter

# higher index = higher security priority (credential access tops the list)
RELATION_PRIORITY = [
    'EXECUTES_PAYLOAD', 'SPAWNS_PROCESS', 'LOADS_SUSPICIOUS_DLL',
    'MODIFIES_REGISTRY', 'PERFORMS_RECONNAISSANCE', 'PERFORMS_PORT_SCAN',
    'MOVES_LATERALLY', 'PERFORMS_BEACONING', 'ESTABLISHES_C2',
    'EXFILTRATES_DATA', 'CAUSES_DENIAL_OF_SERVICE', 'EXPLOITS_VULNERABILITY',
    'BRUTE_FORCES_CREDENTIAL', 'INJECTS_INTO_PROCESS',
    'ACCESS_CREDENTIALS', 'DUMPS_CREDENTIALS',
]
_PRI = {r: i for i, r in enumerate(RELATION_PRIORITY)}

def parent_match(gt_id: str, gen_id: str) -> bool:
    if not gt_id or not gen_id:
        return False
    return gt_id == gen_id or gt_id.split('.')[0] == gen_id.split('.')[0]

def kg_candidate_tids(cid):
    rels = [t.get('relation', '') for t in community_triples.get(str(cid), []) if t.get('relation')]
    if not rels:
        return [], None
    # pick the highest-priority relation present (ties -> first emitted)
    top = max(rels, key=lambda r: (_PRI.get(r, -1), -rels.index(r)))
    return RELATION_TO_TECHNIQUES.get(top, []), top

rows = []
for cid in sorted(community_df['community_id'].unique()):
    g = community_df[community_df['community_id'] == cid]
    gt = g['ground_truth_technique'].mode()
    gt = gt.iloc[0] if len(gt) else None
    cands, top_rel = kg_candidate_tids(cid)
    hit = any(parent_match(gt, c) for c in cands)
    rows.append({'community_id': cid, 'ground_truth': gt, 'top_relation': top_rel,
                 'candidates': cands[:3], 'hit': hit, 'image': g['image'].iloc[0]})

eval_df = pd.DataFrame(rows)
acc = eval_df['hit'].mean() if len(eval_df) else 0.0
print(eval_df.to_string(index=False))
print(f"\nKG technique-recovery accuracy (all processes): {acc:.0%} "
      f"({eval_df['hit'].sum()}/{len(eval_df)})")

# -------------------------------------------------------------------
#  Atomic-dataset reading: the headline metric is whether the MALICIOUS
#  process is recovered, NOT blanket accuracy (every process carries the
#  same dataset-level T1003.001 label, incl. benign Explorer/csrss/cmd).
# -------------------------------------------------------------------
mal = eval_df[eval_df['image'].str.contains('Dumpert', case=False)]
if len(mal):
    ok = mal['hit'].all()
    print(f"\n>>> MALICIOUS-PROCESS RESULT (Outflank-Dumpert.exe): "
          f"{'CORRECTLY mapped to T1003.001' if ok else 'MISSED'}")
    print(f"    top relation extracted: {mal['top_relation'].iloc[0]} "
          f"-> {mal['candidates'].iloc[0]}")
print("\nReport the malicious-process result for the atomic prototype; "
      "report macro-accuracy only at APT29 scale where multiple distinct "
      "ground-truth techniques exist.")

# -------------------------------------------------------------------
#  EXTRACTION DIAGNOSTIC — separates "LLM missed the signal" from
#  "scoring buried it". Flags any alert whose TEXT clearly states LSASS
#  credential access but whose extracted triples lack DUMPS_CREDENTIALS.
# -------------------------------------------------------------------
print("\n--- extraction diagnostic ---")
for cid in sorted(community_df['community_id'].unique()):
    g = community_df[community_df['community_id'] == cid]
    text = g['semantic_alert'].iloc[0].lower()
    rels = [t.get('relation','') for t in community_triples.get(str(cid), [])]
    says_lsass = 'lsass' in text and 'credential' in text
    got_dump   = 'DUMPS_CREDENTIALS' in rels
    if says_lsass and not got_dump:
        print(f"  [{g['image'].iloc[0]}] alert states LSASS credential access "
              f"but LLM extracted {rels} — EXTRACTION MISS (prompt/model issue).")
    if got_dump and not says_lsass:
        print(f"  [{g['image'].iloc[0]}] LLM emitted DUMPS_CREDENTIALS but alert "
              f"does not mention LSASS — HALLUCINATION.")


 community_id ground_truth         top_relation                candidates   hit                image
            0    T1003.001    DUMPS_CREDENTIALS [T1003.001, T1003, T1555]  True Outflank-Dumpert.exe
            1    T1003.001 INJECTS_INTO_PROCESS        [T1055, T1055.001] False          svchost.exe
            2    T1003.001    MODIFIES_REGISTRY        [T1112, T1547.001] False         Explorer.EXE
            3    T1003.001     EXFILTRATES_DATA            [T1041, T1048] False          svchost.exe
            4    T1003.001 INJECTS_INTO_PROCESS        [T1055, T1055.001] False              cmd.exe
            5    T1003.001 INJECTS_INTO_PROCESS        [T1055, T1055.001] False          conhost.exe
            6    T1003.001 INJECTS_INTO_PROCESS        [T1055, T1055.001] False            csrss.exe
            7    T1003.001    MODIFIES_REGISTRY        [T1112, T1547.001] False            lsass.exe

KG technique-recovery accuracy (all processes): 12% (1/8)

>>> MALICIOUS-PROCESS RESULT (O

In [9]:
# =====================================================================
#  CELL G — (OPTIONAL) RAG REPORT for the malicious community
#  ---------------------------------------------------------------------
#  Reuses your ChromaDB collection + generate_rag_report. Only run after
#  CELL 35's ChromaDB build has executed in this kernel. This confirms the
#  end-to-end RAG path produces a grounded report on host data.
# =====================================================================
def run_rag_demo():
    if 'collection' not in globals():
        print("Run the ChromaDB build cell (CIC-IDS cell 35) first.")
        return
    # pick the community whose dominant relation is credential dumping
    target_cid = None
    for cid in community_df['community_id'].unique():
        rels = [t.get('relation','') for t in community_triples.get(str(cid), [])]
        if 'DUMPS_CREDENTIALS' in rels:
            target_cid = cid
            break
    if target_cid is None:
        print("No credential-dumping community found — check extraction output.")
        return
    scope = 'credential-access'
    q = "process accessing LSASS memory to extract credentials"
    res = collection.query(query_texts=[q], n_results=5,
                           where={'tactic': scope})
    print(f"Community {target_cid} — top ChromaDB hits (scope={scope}):")
    for tid, doc in zip(res['ids'][0], res['documents'][0]):
        print(f"  {tid}: {doc.splitlines()[1]}")  # the Name line

run_rag_demo()

Run the ChromaDB build cell (CIC-IDS cell 35) first.
